In [94]:
import pandas as pd
import pyomo.environ as pe
import pyomo.opt as po

In [95]:
m = pe.ConcreteModel("Strength Training Optimization Problem")

### CLEANING DATASET

In [96]:
df  = pd.read_csv("gym_exercise_dataset.csv")
df = df[["Exercise Name","Difficulty (1-5)","Utility","Main_muscle", "Secondary Muscles","Target_Muscles"]]
df["Utility"] = df["Utility"].apply(lambda x: 1 if "Basic" == x else 0)

df.to_csv("cleaned_gym_exercise_dataset.csv", index=False)
muscle_duration = {
    "Neck": 1.5,
    "Calves": 1.5,
    "Hips": 2.0,
    "Back" : 5.0,
    "Thighs": 5.0,
    "Chest": 5.0,
    "Shoulder": 3.5,
    "Upper Arms": 3.0,
    "Forearm": 2.0,
    "Biceps": 3.0,
    "Triceps": 3.0,
}



def refine_upper_arms(main_muscle: str, target_muscle: str):
    """
    If main_muscle == 'Upper Arms', use target_muscle to classify into Biceps or Triceps.
    Otherwise return main_muscle unchanged.
    """

    if main_muscle != "Upper Arms":
        return main_muscle

    if not isinstance(target_muscle, str):
        return "Upper Arms"

    text = target_muscle.lower().strip(",")

    # --- Biceps indicators ---
    BICEPS_KEYS = [
        "biceps brachii",
        "brachialis",
        "brachioradialis",
    ]

    # --- Triceps indicators ---
    TRICEPS_KEYS = [
        "triceps brachii",
    ]
    
    # Classification
    for key in BICEPS_KEYS:
        if key in text:
            return "Biceps"

    for key in TRICEPS_KEYS:
        if key in text:
            return "Triceps"

    # If it is upper arms but no match -> leave as Upper Arms (or return None)
    return "Upper Arms"
df["Main_muscle"] = df.apply(
    lambda row: refine_upper_arms(row["Main_muscle"], row["Target_Muscles"]),
    axis=1
)
df["Main_muscle"].unique().tolist()

def map_secondary_to_main(secondary_muscle: str):
    if not isinstance(secondary_muscle, str):
        return None

    # Mapping from keywords to main muscle categories
    MUSCLE_MAP = {
        # Neck
        "sternocleidomastoid": "Neck",
        "levator scapulae": "Neck",
        "splenius": "Neck",

        # Shoulder
        "deltoid": "Shoulder",
        "supraspinatus": "Shoulder",
        "infraspinatus": "Shoulder",
        "teres minor": "Shoulder",
        "subscapularis": "Shoulder",

        # Upper arms
        "biceps": "Biceps",
        "triceps": "Triceps",
        "brachialis": "Biceps",
        "brachioradialis": "Biceps",

        # Back
        "trapezius": "Back",
        "rhomboids": "Back",
        "latissimus dorsi": "Back",
        "erector spinae": "Back",
        
        # Chest
        "pectoralis": "Chest",

        # Hips
        "gluteus maximus": "Hips",
        "iliopsoas": "Hips",
        "hip flexors": "Hips",

        # Thighs
        "quadriceps": "Thighs",
        "hamstrings": "Thighs",
        "adductors": "Thighs",

        # Calves
        "gastrocnemius": "Calves",
        "soleus": "Calves",
        "tibialis anterior": "Calves",
    }

    # Normalize input
    text = secondary_muscle.lower().strip()

    # Search for any keyword
    for keyword, main_category in MUSCLE_MAP.items():
        if keyword in text:
            return main_category

    return None
df["Secondary Muscles"] = df["Secondary Muscles"].apply(map_secondary_to_main)
df["Duration"] = df["Main_muscle"].map(muscle_duration)
df.loc[df["Main_muscle"] == df["Secondary Muscles"], "Secondary Muscles"] = None
df = df.drop(columns=["Target_Muscles"])
df.drop_duplicates(inplace=True)
df.to_csv("cleaned_gym_exercise_dataset.csv", index=False)
df.duplicated().value_counts()

False    372
Name: count, dtype: int64

### SETS

In [97]:
# ========= SETS =========
EXERCISES = df["Exercise Name"].tolist()

MAIN_MUSCLES = df["Main_muscle"].unique().tolist()

SECONDARY_MUSCLES = df["Secondary Muscles"].unique().tolist()
DAYS = list(range(1, 29)) 

# ========= PARAMETERS =========
duration = dict(zip(df["Exercise Name"], df["Duration"]))
difficulty = dict(zip(df["Exercise Name"], df["Difficulty (1-5)"]))
main_muscle = dict(zip(df["Exercise Name"], df["Main_muscle"]))
secondary_muscle = dict(zip(df["Exercise Name"], df["Secondary Muscles"]))


# Map muscle → list of exercises
muscle_to_exercises = {
    m: df[df["Main_muscle"] == m]["Exercise Name"].tolist()
    for m in MAIN_MUSCLES
}

In [98]:
m.exercise = pe.Set(initialize=EXERCISES)
m.muscles = pe.Set(initialize=MAIN_MUSCLES)
m.days = pe.Set(initialize=DAYS)

### PARAMETERS

In [99]:
# ========= PARAMETERS =========
duration = dict(zip(df["Exercise Name"], df["Duration"]))
difficulty = dict(zip(df["Exercise Name"], df["Difficulty (1-5)"]))


In [100]:
m.duration = pe.Param(
    m.exercise,
    initialize=duration)

In [101]:
m.difficulty = pe.Param(
    m.exercise,
    initialize=difficulty)

In [102]:
def muscle_exercises_init(df):
    dict = {}
    for e in EXERCISES:
        for m in MAIN_MUSCLES:
            if m in df[df["Exercise Name"] == e]["Main_muscle"].values:
                dict[(e, m)] = 2
            elif m in df[df["Exercise Name"] == e]["Secondary Muscles"].values:
                dict[(e, m)] = 1
            else:
                dict[(e, m)] = 0
    return dict 


muscle_exercises_dict = muscle_exercises_init(df)
print(muscle_exercises_dict)
m.muscle_group_exercise = pe.Param(
    m.exercise,m.muscles,
    initialize=muscle_exercises_dict)

{('Neck Flexion', 'Neck'): 2, ('Neck Flexion', 'Shoulder'): 0, ('Neck Flexion', 'Triceps'): 0, ('Neck Flexion', 'Biceps'): 0, ('Neck Flexion', 'Forearm'): 0, ('Neck Flexion', 'Back'): 0, ('Neck Flexion', 'Chest'): 0, ('Neck Flexion', 'Hips'): 0, ('Neck Flexion', 'Thighs'): 0, ('Neck Flexion', 'Calves'): 0, ('Lateral Neck Flexion', 'Neck'): 2, ('Lateral Neck Flexion', 'Shoulder'): 0, ('Lateral Neck Flexion', 'Triceps'): 0, ('Lateral Neck Flexion', 'Biceps'): 0, ('Lateral Neck Flexion', 'Forearm'): 0, ('Lateral Neck Flexion', 'Back'): 0, ('Lateral Neck Flexion', 'Chest'): 0, ('Lateral Neck Flexion', 'Hips'): 0, ('Lateral Neck Flexion', 'Thighs'): 0, ('Lateral Neck Flexion', 'Calves'): 0, ('Wall Front Neck Bridge', 'Neck'): 2, ('Wall Front Neck Bridge', 'Shoulder'): 0, ('Wall Front Neck Bridge', 'Triceps'): 0, ('Wall Front Neck Bridge', 'Biceps'): 0, ('Wall Front Neck Bridge', 'Forearm'): 0, ('Wall Front Neck Bridge', 'Back'): 0, ('Wall Front Neck Bridge', 'Chest'): 0, ('Wall Front Neck B

In [103]:
#Si es basico o no
m.exercise_basic = pe.Param(
    m.exercise,
    initialize=dict(zip(df["Exercise Name"], df["Utility"]))
)
print(dict(zip(df["Exercise Name"], df["Utility"])))

{'Neck Flexion': 0, 'Lateral Neck Flexion': 0, 'Wall Front Neck Bridge': 0, 'Wall Side Neck Bridge': 0, 'Neck Extension': 0, 'Seated Neck Extension': 0, 'Seated Neck Extension:  Harness': 0, 'Neck Retraction': 0, 'Wall Rear Neck Bridge': 0, 'Lying Neck Retraction': 0, 'Front Raise': 0, 'Military Press': 1, 'Military Press:  Seated': 1, 'Front Raise:  Alternating': 0, 'Front Raise:  One Arm': 0, 'Shoulder Press': 1, 'Shoulder Press:  Seated': 0, 'Arnold Press': 0, 'Shoulder Press:  One Arm': 0, 'Reclined Shoulder Press': 1, 'Shoulder Press:  Parallel Grip': 1, 'Pike Press (between benches)': 1, 'Pike Press (between benches):  Elevated (between benches)': 1, 'Upright Row': 1, 'Lateral Raise': 0, 'Lateral Raise:  One Arm': 0, 'Upright Row:  One Arm': 1, 'Upright Row:  with rope': 1, 'Y Raise': 0, 'Incline Lateral Raise': 0, 'Lateral Raise:  other machine': 0, 'Upright Row\u200b\u200b\u200b\u200b\u200b\u200b\u200b': 1, 'Rear Delt Row': 0, 'Reverse Fly': 0, 'Rear Delt Row:  Standing Rear De

In [104]:
diccionary_series_per_muscle = {
    "Neck": 6,
    "Calves": 12,
    "Hips": 12,
    "Back" : 20,
    "Thighs": 12,
    "Chest": 20,
    "Shoulder": 15,
    "Forearm": 8,
    "Biceps": 16,
    "Triceps": 16,
} #Esto deberia de modificarse para que tenga mas logica.

m.series_per_muscle = pe.Param(
    m.muscles,
    initialize=diccionary_series_per_muscle
)

In [105]:
NUMBER_DAYS_PER_MONTH = 16
m.number_days_per_month = pe.Param(
    initialize=NUMBER_DAYS_PER_MONTH
)

In [106]:
DURATION_SESSION_MAX = 90
m.duration_session_max = pe.Param(
    initialize=DURATION_SESSION_MAX
)

### VARIABLES

In [107]:
m.training_day = pe.Var(m.days,domain=pe.Binary)
m.sets_day = pe.Var(m.days, m.exercise, domain=pe.PositiveIntegers)
m.positive_deviation = pe.Var(m.muscles, domain=pe.PositiveIntegers)
m.negative_deviation = pe.Var(m.muscles, domain=pe.PositiveIntegers)

### OBJECTIVE FUNCTION

In [108]:
def obj_func(model):
    cost = 0
    for muscle in model.muscles:
        cost +=  model.positive_deviation[muscle] + model.negative_deviation[muscle]
    return cost

m.cost=pe.Objective(rule=obj_func(m),sense=pe.minimize)

 ### CONSTRAINTS

In [109]:
m.number_a_month = pe.ConstraintList()

m.number_a_month.add(
                sum(m.training_day[day] for day in m.days) <= m.number_days_per_month
            )


## Solver


In [110]:
solver = po.SolverFactory('gurobi')
results = solver.solve(m, tee=True)

Set parameter Username
Set parameter LicenseID to value 2707532


GurobiError: Version number is 13.0, license is for version 12.0